## Paso 0 — Clonar el repositorio

Esta celda es necesaria si abres este notebook directo desde GitHub "Open in Colab" — eso solo carga el .ipynb suelto, no trae la carpeta `data/` que necesita el resto del notebook. Correr esto primero soluciona eso.

In [ ]:
# Clona el repo para tener acceso a data/ (los .json/.jsonl del pipeline)
# IMPORTANTE: verifica que esta URL sea la de tu repo real antes de correr
!git clone https://github.com/alexahurtado08/TravelGenie_AlexandraHurtado_MarianaValderrama.git
%cd TravelGenie_AlexandraHurtado_MarianaValderrama
!ls data/  # confirma que aparezcan los .json/.jsonl, no solo el .txt de aviso

# Fine-tuning con LoRA

A partir de aquí, todo corre sobre `data/07_train.jsonl` y `data/07_val.jsonl` generados arriba.

**Resumen de lo que hacen las 5 celdas:**
1. Instalación de librerías + configuración (modelo base, semilla fijada).
2. Carga y tokenización del dataset.
3. Evaluación del **baseline**: `flan-t5-small` sin fine-tuning (zero-shot), mismo prompt.
4. Configuración de **LoRA** y entrenamiento.
5. Evaluación del modelo afinado, tabla comparativa contra el baseline, y 3 ejemplos cualitativos.

Modelo base: `google/flan-t5-small` (encoder-decoder). Se eligió esta familia porque la tarea central (redactar una recomendación en lenguaje natural a partir de datos estructurados) es *data-to-text* — el objetivo de preentrenamiento text-to-text de T5 ya entrena exactamente ese tipo de mapeo, a diferencia de un encoder-only (no genera texto) o un decoder-only (mezcla comprensión y generación en un solo bloque, menos natural para este input corto/output largo).

### Celda 1 — Instalación e imports

Instala/actualiza `transformers`, `peft`, `torchao` (la versión que trae Colab por defecto es incompatible con `peft` reciente, hay que forzar la actualización), fija la semilla (`SEED = 42`, reproducibilidad) y define las constantes del experimento.

In [15]:
#celda 1
!pip install -q -U transformers peft accelerate evaluate rouge_score datasets torchao

import json
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments, Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
)
from peft import LoraConfig, get_peft_model, TaskType
import evaluate

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

MODEL_NAME = "google/flan-t5-small"
PREFIJO_TAREA = "redacta una recomendación de viaje con estos datos: "
MAX_INPUT_LEN = 220  # subido de 64: el input ahora incluye el bloque de hechos (clima, vuelo, lugares)
MAX_TARGET_LEN = 256

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Usando:", device)

Usando: cuda


### Celda 2 — Cargar y tokenizar el dataset

Carga `07_train.jsonl` / `07_val.jsonl` con 🤗 `datasets`, y tokeniza cada par con el tokenizer de `flan-t5-small`. `MAX_INPUT_LEN=220` (subido de 64 en una iteración anterior) porque el input ahora incluye el bloque completo de hechos, no solo la pregunta.

In [16]:
#celda2
dataset = load_dataset("json", data_files={
    "train": "data/07_train.jsonl",
    "validation": "data/07_val.jsonl",
})
print(dataset)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocesar(ejemplo):
    entrada = PREFIJO_TAREA + ejemplo["input"]
    modelo_in = tokenizer(entrada, max_length=MAX_INPUT_LEN, truncation=True)
    etiquetas = tokenizer(text_target=ejemplo["output"], max_length=MAX_TARGET_LEN, truncation=True)
    modelo_in["labels"] = etiquetas["input_ids"]
    return modelo_in

dataset_tok = dataset.map(preprocesar, remove_columns=["input", "output"])

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input', 'output'],
        num_rows: 276
    })
    validation: Dataset({
        features: ['input', 'output'],
        num_rows: 70
    })
})


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

...

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

Map:   0%|          | 0/276 [00:00<?, ? examples/s]

Map:   0%|          | 0/70 [00:00<?, ? examples/s]

### Celda 3 — Evaluar el baseline (zero-shot)

Carga `flan-t5-small` **sin ningún fine-tuning** y genera respuestas para todo el conjunto de validación, con el mismo prompt que se usará después para el modelo afinado. Este es el punto de comparación obligatorio de la rúbrica: mismo modelo, mismo prompt, mismo conjunto — cualquier diferencia en las métricas se explica solo por el fine-tuning.

`no_repeat_ngram_size`, `repetition_penalty` y `min_new_tokens` en la generación no son arbitrarios: se agregaron después de que una corrida temprana del modelo afinado colapsara en loops de repetición ("clima, clima, clima...") — quedan también en la evaluación del baseline para que la comparación sea justa (mismos parámetros de generación en ambos lados).

In [17]:
#Celda 3
rouge = evaluate.load("rouge")

def cargar_val_crudo(path="data/07_val.jsonl"):
    ejemplos = []
    with open(path, encoding="utf-8") as f:
        for linea in f:
            ejemplos.append(json.loads(linea))
    return ejemplos

def evaluar_modelo(modelo, ejemplos, etiqueta=""):
    modelo.eval()
    predicciones, referencias = [], []
    with torch.no_grad():
        for ej in ejemplos:
            entrada = PREFIJO_TAREA + ej["input"]
            ids = tokenizer(entrada, return_tensors="pt", max_length=MAX_INPUT_LEN, truncation=True).to(device)
            salida = modelo.generate(
                **ids,
                max_length=MAX_TARGET_LEN,
                min_new_tokens=60,        # fuerza a no cortar demasiado corto
                num_beams=4,
                no_repeat_ngram_size=3,   # prohíbe repetir el mismo trigrama (frena los loops)
                repetition_penalty=1.3,   # penaliza reusar tokens ya generados
                early_stopping=True,
            )
            texto = tokenizer.decode(salida[0], skip_special_tokens=True)
            predicciones.append(texto)
            referencias.append(ej["output"])

    resultado = rouge.compute(predictions=predicciones, references=referencias)
    print(f"\\nROUGE ({etiqueta}):")
    for k, v in resultado.items():
        print(f"  {k}: {v:.4f}")
    return resultado, predicciones, referencias

modelo_base = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
val_ejemplos = cargar_val_crudo()

print("=== Evaluando BASELINE (zero-shot, sin fine-tuning) ===")
rouge_baseline, preds_baseline, refs = evaluar_modelo(modelo_base, val_ejemplos, etiqueta="baseline zero-shot")

model.safetensors: reconstructing file:   0%|          |  0.00B /  308MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

=== Evaluando BASELINE (zero-shot, sin fine-tuning) ===
\nROUGE (baseline zero-shot):
  rouge1: 0.3526
  rouge2: 0.1992
  rougeL: 0.2863
  rougeLsum: 0.2863


### Celda 4 — Configurar LoRA y entrenar

Envuelve `flan-t5-small` con un adaptador LoRA y lo entrena sobre `dataset_tok["train"]`. Los hiperparámetros (`r=8`, `lora_alpha=16`, `target_modules=["q","v"]`, `learning_rate=5e-4`, `epochs=12`) no se eligieron a la primera intentona — están justificados en el comentario del código con el historial real de 3 corridas (LR alto → colapso en loops; LR bajo → subaprendizaje; este punto medio es el que funcionó).

In [18]:
#Celda 4
# Justificación de hiperparámetros (para el README):
# - r=8: rank bajo, apropiado para un dataset pequeño (78 ejemplos train) —
#   un rank alto arriesga sobreajustar memorizando las plantillas.
# - lora_alpha=16 (2x el rank): heurística estándar de la literatura de LoRA.
# - target_modules=["q","v"]: proyecciones de atención query/value del
#   encoder-decoder T5 — es el punto recomendado por el paper original de
#   LoRA para mantener pocos parámetros entrenables sin perder capacidad
#   de adaptación.
# - learning_rate=5e-4, epochs=12: se probaron 3 configuraciones antes de esta.
#   1e-3 / 15 épocas colapsó en loops de repetición ("clima, clima, clima...").
#   3e-4 / 8 épocas subaprendió (respuestas demasiado cortas/genéricas).
#   5e-4 / 12 épocas es el punto medio que efectivamente funcionó (ver README).
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q", "v"],
)

modelo_lora = get_peft_model(AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device), lora_config)
modelo_lora.print_trainable_parameters()

data_collator = DataCollatorForSeq2Seq(tokenizer, model=modelo_lora)

args = Seq2SeqTrainingArguments(
    output_dir="./travelgenie-lora",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=5e-4,           # punto medio: 1e-3 colapsó en loops, 3e-4 subaprendió
    num_train_epochs=12,          # idem, punto medio entre 15 (sobreajuste) y 8 (corto)
    weight_decay=0.01,            # regularización extra
    eval_strategy="epoch",
    save_strategy="no",
    predict_with_generate=True,
    logging_steps=5,
    seed=SEED,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=modelo_lora,
    args=args,
    train_dataset=dataset_tok["train"],
    eval_dataset=dataset_tok["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
)

trainer.train()

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


trainable params: 344,064 || all params: 77,305,216 || trainable%: 0.4451


Epoch,Training Loss,Validation Loss
1,1.964682,1.581593
2,1.397720,0.988865
3,0.966129,0.574664
4,0.704654,0.370864
5,0.573811,0.276010
6,0.469382,0.196514
7,0.390756,0.144310
8,0.354587,0.112726
9,0.312486,0.094348
10,0.286229,0.079988


TrainOutput(global_step=420, training_loss=0.75036313022886, metrics={'train_runtime': 129.4075, 'train_samples_per_second': 25.594, 'train_steps_per_second': 3.246, 'total_flos': 249000884379648.0, 'train_loss': 0.75036313022886, 'epoch': 12.0})

### Celda 5 — Evaluación final y ejemplos cualitativos

Evalúa el modelo ya afinado con LoRA sobre el mismo conjunto de validación, imprime la tabla comparativa ROUGE (baseline vs. LoRA, con el delta) que va al README, y muestra 3 ejemplos de entrada→salida comparando baseline, LoRA y la referencia real — lo que pide explícitamente la rúbrica.

In [19]:
#Celda 5
print("=== Evaluando MODELO CON FINE-TUNING (LoRA) ===")
rouge_lora, preds_lora, _ = evaluar_modelo(modelo_lora, val_ejemplos, etiqueta="con fine-tuning LoRA")

print("\\n=== TABLA COMPARATIVA (para el README) ===")
print(f"{'Métrica':<12}{'Baseline':<12}{'Con LoRA':<12}{'Delta':<10}")
for k in rouge_baseline:
    base, lora = rouge_baseline[k], rouge_lora[k]
    print(f"{k:<12}{base:<12.4f}{lora:<12.4f}{lora-base:+.4f}")

print("\\n=== 3 EJEMPLOS CUALITATIVOS ===")
for i in range(3):
    print(f"\\n--- Ejemplo {i+1} ---")
    print("INPUT:    ", val_ejemplos[i]["input"])
    print("BASELINE: ", preds_baseline[i])
    print("LORA:     ", preds_lora[i])
    print("REAL:     ", val_ejemplos[i]["output"])

=== Evaluando MODELO CON FINE-TUNING (LoRA) ===
\nROUGE (con fine-tuning LoRA):
  rouge1: 0.6041
  rouge2: 0.4579
  rougeL: 0.5156
  rougeLsum: 0.5153
\n=== TABLA COMPARATIVA (para el README) ===
Métrica     Baseline    Con LoRA    Delta     
rouge1      0.3526      0.6041      +0.2515
rouge2      0.1992      0.4579      +0.2587
rougeL      0.2863      0.5156      +0.2293
rougeLsum   0.2863      0.5153      +0.2290
\n=== 3 EJEMPLOS CUALITATIVOS ===
\n--- Ejemplo 1 ---
INPUT:     Datos: Destino: Riohacha, La Guajira. Mejor mes por clima: diciembre. Vuelo más económico: agosto desde BOG, $223,222 COP. Hoteles: Castillo del Mar, RIOHACHA, casa. Restaurantes: Los Montaditos, Wow Pizza, Asadero Emir. Atractivos: Muelle peatonal, Comunidad La Raya. Pregunta: Quiero ir a Riohacha, ¿cuándo me conviene viajar?
BASELINE:  recomendación de viaje con estas datos: Datos: Destino: Riohacha, La Guajira. Mejor mes por clima: diciembre. Vuelo más económico: agosto desde BOG, $223,222 COP. Hoteles: Cast